# Murmur Simulation Pipeline -- Deep Dive

This notebook walks through the **entire simulation pipeline** end to end. It shows:
1. Where the data comes from (the survey + schema system)
2. How that data becomes context for the AI (RAG builder + research library)
3. How the ML calibration model works
4. How personas are generated (freeform vs manifest-constrained)
5. How each persona is interviewed
6. How responses are aggregated into a final report

Every code cell loads and displays the **actual files and data structures** used in production. Nothing is mocked -- this is the real system.

In [ ]:
# Setup: add project root to path so we can import backend modules
import sys, os, json, yaml
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path(os.getcwd()).parent  # murmur/murmur/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Key directories:")
for d in ["backend/swarm", "backend/context", "backend/reviewer_intelligence", 
          "backend/survey", "backend/ml", "config", "research"]:
    exists = (PROJECT_ROOT / d).exists()
    print(f"  {d}/ {'OK' if exists else 'MISSING'}")

---
# Part 1: The Full Pipeline at a Glance

Before diving into each piece, here is the complete data flow. Every simulation goes through these steps in order:

```
USER FILLS SURVEY (43 fields across 3 sections)
        |
        v
+-------+--------+------------------+
|                |                   |
v                v                   v
RAG Builder      Feature Extractor   Research Library
(text context)   (23 ML features)    (Hofstede + psych)
|                |                   |
v                v                   v
Survey Context   ML Calibration      Research Context
(~2000 chars)    (accuracy est.)     (~4000 chars)
|                                    |
+----------------+-------------------+
                 |
                 v
    Context Enrichment Engine (8 live tools)
    Claude picks 2-5 tools -> runs in parallel
    -> filters into narrative
                 |
                 v
    ENRICHED CONTEXT NARRATIVE (all sources merged)
                 |
        +--------+--------+
        |                 |
        v                 v
    Reviewer Intel    Persona Generation
    (Google Reviews   (15 personas, each with
    -> bias correct   name/age/personality/
    -> 6 segments     visit pattern/quirks)
    -> manifest)           |
                           v
                    Simulation (parallel interviews)
                    Each persona answers independently
                    via Claude API (semaphore=5)
                           |
                           v
                    Aggregation + Impact + Caveats
                    -> headline, themes, standout voices
                    -> revenue CI, decision framework
                    -> research-backed warnings
                           |
                           v
                    RESULTS STORED IN SUPABASE
```

---
# Part 2: Where the Data Comes From -- The Survey Schema

Everything starts with the user filling out a survey about their business. The survey is defined in a single YAML file: `config/survey_schema.yaml`. This file is the **single source of truth** -- it controls:

- What questions appear in the frontend form
- How each answer feeds into the RAG context (text for the AI)
- How each answer becomes an ML feature (number for the model)
- How each answer contributes to profile completeness scoring

Let's look at it.

In [ ]:
# Load the raw survey schema
schema_path = PROJECT_ROOT / "config" / "survey_schema.yaml"
with open(schema_path) as f:
    raw_schema = yaml.safe_load(f)

# Show the top-level structure
print("Schema version:", raw_schema.get("version"))
print("Target:", raw_schema.get("target"))
print()
print("Sections:")
for section in raw_schema["sections"]:
    fields = section["fields"]
    print(f"  {section['id']}: {section['label']} ({len(fields)} fields)")
    for field in fields[:3]:
        print(f"    - {field['id']}: {field['label']} ({field['type']})")
    if len(fields) > 3:
        print(f"    ... and {len(fields) - 3} more fields")

In [ ]:
# Let's look at ONE field in detail to see how much metadata each carries.
# "location_country" has the highest accuracy weight (8) -- it unlocks Hofstede cultural data.

example_field = None
for section in raw_schema["sections"]:
    for field in section["fields"]:
        if field["id"] == "location_country":
            example_field = field
            break

print("=== Example Field: location_country ===")
print()
print(f"ID:    {example_field['id']}")
print(f"Label: {example_field['label']}")
print(f"Type:  {example_field['type']}")
print()

print("STORAGE config (where it gets saved):")
pprint(example_field.get("storage", {}))
print()

print("RAG config (how it becomes text context for the AI):")
pprint(example_field.get("rag", {}))
print()

print("ML config (how it becomes a number for the model):")
pprint(example_field.get("ml", {}))
print()

print("ACCURACY config (how much it matters for profile completeness):")
pprint(example_field.get("accuracy", {}))

In [ ]:
# Show all fields ranked by accuracy weight -- this tells us which questions
# matter most for simulation quality.

all_fields = []
for section in raw_schema["sections"]:
    for field in section["fields"]:
        weight = field.get("accuracy", {}).get("weight", 0)
        rag_included = field.get("rag", {}).get("include", False)
        ml_included = field.get("ml", {}).get("include", False)
        all_fields.append({
            "field": field["id"],
            "weight": weight,
            "rag": rag_included,
            "ml": ml_included,
            "type": field["type"],
        })

all_fields.sort(key=lambda x: x["weight"], reverse=True)

print(f"{'Field':<30} {'Weight':>6} {'RAG':>5} {'ML':>5} {'Type':<15}")
print("-" * 75)
for f in all_fields:
    print(f"{f['field']:<30} {f['weight']:>6} {'yes' if f['rag'] else '-':>5} {'yes' if f['ml'] else '-':>5} {f['type']:<15}")
print()
print(f"Total fields: {len(all_fields)}")
print(f"Fields used in RAG context: {sum(1 for f in all_fields if f['rag'])}")
print(f"Fields used in ML model: {sum(1 for f in all_fields if f['ml'])}")

## What a filled-out survey looks like

Here is an example of what the survey data looks like when a user fills it out. This is one of the hardcoded test profiles from `backend/ml/training_data.py` -- a Spanish restaurant in Madrid.

In [ ]:
# A realistic example: what a Spanish restaurant owner would fill in.
# This is one of 8 known test profiles from backend/ml/training_data.py.

example_survey = {
    "name": "La Tasca de Miguel",
    "type": "restaurant",
    "description": "Traditional Spanish tapas restaurant in the Gothic Quarter. "
                   "Family recipes, local wines, no-frills atmosphere.",
    "location_country": "ES",
    "location_city": "Barcelona",
    "location_neighbourhood": "Gothic Quarter",
    "years_open": "3-10",
    "business_role": "habit",            # customers come out of routine
    "visit_frequency": "weekly",         # how often regulars come
    "customer_value_drivers": ["atmosphere", "personal_touch", "quality"],
    "customer_social_context": ["with_friends", "date_night"],
    "regular_proportion": "solid_base",  # decent number of regulars
    "area_demographics": ["business", "tourist"],
    "competitor_count": "three_five",    # 3-5 similar businesses nearby
    "area_feel": "community",            # neighbourhood vibe
    "customer_description": "Local office workers, regulars who come for lunch, "
                           "some tourists on weekends. Price conscious.",
}

print("=== Example Survey Data ===")
print()
for key, value in example_survey.items():
    print(f"  {key}: {value}")

---
# Part 3: How Survey Data Becomes AI Context (RAG Builder)

The RAG builder reads each survey field's `rag.template` from the schema and renders it with the user's answer. The result is a structured text document that gets injected into every Claude prompt via the `{{context_narrative}}` template variable.

This is the bridge between "data the user typed in a form" and "text the AI reads to understand the business."

In [ ]:
# Build RAG context from the example survey data.
# This is the actual function used in the simulation pipeline.

from backend.survey.rag_builder import build_rag_context

rag_text = build_rag_context(example_survey)

print("=== RAG Context Output ===")
print(f"(Total length: {len(rag_text)} characters)")
print()
print(rag_text)

## Research Library Context

In addition to the survey RAG, we also inject research-backed knowledge. The research library provides:
- **Hofstede cultural dimensions** for the business's country (power distance, individualism, uncertainty avoidance, etc.)
- **Consumer psychology** insights (loss aversion, anchoring, status quo bias)
- **Review bias** research (why reviews are not representative)

This grounds the AI's persona generation in peer-reviewed behavioral science.

In [ ]:
# Load the research context for Spain
from research.rag_library import get_country_profile, get_simulation_context, get_persona_cultural_modifier

# 1. Hofstede cultural profile
profile = get_country_profile("ES")
print("=== Hofstede Profile for Spain (ES) ===")
if profile:
    for key, value in profile.items():
        if key != "simulation_modifiers":
            print(f"  {key}: {value}")
    print()
    print("  Simulation modifiers:")
    for mod_key, mod_val in profile.get("simulation_modifiers", {}).items():
        print(f"    {mod_key}: {mod_val}")
else:
    print("  (No profile found -- would fall back to defaults)")

print()

# 2. Cultural modifier string (injected into persona prompts)
modifier = get_persona_cultural_modifier("ES")
print("=== Cultural Modifier for Persona Prompts ===")
print(modifier if modifier else "(No modifier available)")

print()

# 3. Full simulation context (truncated for display)
full_context = get_simulation_context("ES", "consumer")
print(f"=== Full Research Context ({len(full_context)} chars) ===")
print(full_context[:1500])
print(f"\n... [{len(full_context) - 1500} more characters]")

## Context Enrichment Engine (Live API tools)

On top of the static survey + research context, the pipeline runs **live API tools** to gather real-time market intelligence. Claude acts as an orchestrator -- it reads the business profile and question, then decides which 2-5 tools to run.

The 8 available tools:

| Tool | What it does | API |
|------|-------------|-----|
| `web_search` | General web search for market context | Brave Search |
| `google_places` | Find nearby competitors, ratings, reviews | Google Places |
| `news_search` | Recent news about the area/industry | Brave News |
| `review_analyzer` | Extract themes from the business's own reviews | Google Places + Claude |
| `price_index` | Regional price/inflation data | Eurostat / World Bank |
| `weather_trends` | Seasonal weather patterns (for outdoor businesses) | Open-Meteo |
| `demographic` | Population, income, urbanization stats | World Bank |
| `social_sentiment` | Reddit discussions about the area/industry | Reddit API |

**Key design decisions:**
- The orchestrator is a single Claude API call that returns a JSON plan
- Tools run in parallel with a 30s per-tool timeout and 90s global timeout
- If >3 tools return data, a second Claude call filters for relevance
- `gather_context()` **NEVER raises** -- if everything fails, the simulation proceeds with just the survey data
- The output is a plain text narrative, not structured JSON

In [ ]:
# Let's look at how the orchestrator decides which tools to run.
# This is the actual prompt template it uses.

orchestrator_prompt_path = PROJECT_ROOT / "backend" / "context" / "prompts" / "orchestrator.txt"
orchestrator_prompt = orchestrator_prompt_path.read_text()

print("=== Orchestrator Prompt (what Claude sees to plan research) ===")
print(f"({len(orchestrator_prompt)} characters)")
print()
print(orchestrator_prompt[:2000])
if len(orchestrator_prompt) > 2000:
    print(f"\n... [{len(orchestrator_prompt) - 2000} more characters]")

In [ ]:
# How all context sources get merged in the simulation pipeline.
# This is the actual merging logic from backend/api/routes/simulations.py:

# In the real pipeline, _run_pipeline() does this:
survey_context = rag_text                                    # from RAG builder
research_context = full_context[:4000]                       # from research library (truncated)
agent_context = "(live agent results would go here)"         # from context engine

# All three are joined with separators
enriched_parts = []
if survey_context:
    enriched_parts.append(survey_context)
if research_context:
    enriched_parts.append(research_context)
if agent_context:
    enriched_parts.append(agent_context)

combined_narrative = "\n\n---\n\n".join(enriched_parts)

print(f"=== Combined Context Narrative ===")
print(f"Survey context:   {len(survey_context):>6} chars")
print(f"Research context: {len(research_context):>6} chars")
print(f"Agent context:    {len(agent_context):>6} chars (placeholder)")
print(f"{'':->40}")
print(f"Total combined:   {len(combined_narrative):>6} chars")
print()
print("This entire text blob gets injected into the {{context_narrative}}")
print("template variable in ALL THREE prompt templates:")
print("  1. persona_base.txt     (persona generation)")
print("  2. persona_interview.txt (each persona's interview)")
print("  3. aggregation.txt       (final synthesis)")

---
# Part 4: The ML Calibration Model

The ML model does NOT run the simulation. It answers a meta-question: **"Given this business profile, how confident should we be in the simulation's accuracy?"**

It works like this:
1. `FeatureExtractor` converts survey answers into 23 numerical features (one-hot encoding, ordinal mapping, Hofstede-derived scores)
2. `ModelArena` trains 6 models (RandomForest, XGBoost, CatBoost, LightGBM, LogisticRegression, GradientBoosting) and picks the best one
3. At simulation time, the winning model predicts P(simulation_is_accurate) and that probability adjusts the confidence score shown to the user

Currently trained on 200 synthetic records from 8 hardcoded profiles. Once we have 50+ real outcomes from users, the model will retrain on actual data.

In [ ]:
# Step 1: Feature extraction -- turning survey answers into numbers

from backend.survey.feature_extractor import FeatureExtractor

extractor = FeatureExtractor()

features = extractor.extract(example_survey)

print("=== ML Features Extracted from Survey ===")
print(f"Total features: {len(extractor.feature_names)}")
print()
print(f"{'Feature Name':<45} {'Value':>8}")
print("-" * 55)
for name in extractor.feature_names:
    val = features.get(name, 0.0)
    # Only show non-zero features to keep it readable
    if val != 0.0:
        print(f"{name:<45} {val:>8.3f}")

print()
zero_count = sum(1 for n in extractor.feature_names if features.get(n, 0) == 0)
print(f"({zero_count} features are zero / not applicable)")

In [ ]:
# Step 2: Training data generation
# The model trains on synthetic data generated from 8 known profiles.

from backend.ml.training_data import generate_training_data, _known_profiles

# Show the 8 base profiles
profiles = _known_profiles()
print(f"=== {len(profiles)} Known Business Profiles for Training ===")
print()
for p in profiles:
    data = p["survey_data"]
    print(f"  {p['name']}")
    print(f"    Type: {data.get('type', '?')}, Country: {data.get('location_country', '?')}, "
          f"City: {data.get('location_city', '?')}")
    print(f"    Base accuracy: {p['base_accuracy']:.0%}")
    print()

print("Each profile gets noised up (random field drops, swaps) to create")
print("~200 synthetic training records. The 'correct' label is sampled")
print("probabilistically based on base_accuracy + profile completeness.")

In [ ]:
# Step 3: Model Arena -- train and compare models
# This takes ~5-10 seconds. We generate fresh data and train all models.

import numpy as np

X, y, feature_names = generate_training_data(n_records=200)

print(f"=== Training Data Shape ===")
print(f"  X: {X.shape} ({X.shape[0]} records x {X.shape[1]} features)")
print(f"  y: {y.shape} ({np.sum(y == 1)} correct, {np.sum(y == 0)} incorrect)")
print(f"  Baseline accuracy: {np.mean(y):.1%} (if we always predict 'correct')")
print()

from backend.ml.calibration_model import ModelArena

arena = ModelArena()
arena.train(X, y, feature_names)

print("=== Model Comparison ===")
print(arena.report())

In [ ]:
# Step 4: What the model tells us at simulation time
# For our Spanish restaurant, what is the predicted accuracy?

feature_vec = [features.get(n, 0.0) for n in extractor.feature_names]
proba = arena.predict_proba(np.array([feature_vec]))
accuracy_estimate = float(proba[0][1]) if len(proba[0]) > 1 else 0.5

print(f"=== ML Calibration for La Tasca de Miguel ===")
print(f"  Best model: {arena.best_model()}")
print(f"  P(simulation is accurate): {accuracy_estimate:.1%}")
print()

# Feature importances -- what survey fields matter most?
importances = arena.feature_importance()
print("=== Top 10 Most Important Features ===")
for i, (feat, imp) in enumerate(list(importances.items())[:10]):
    bar = "#" * int(imp * 100)
    print(f"  {i+1:>2}. {feat:<40} {imp:.3f} {bar}")

print()
print("These importances feed back into the survey schema's accuracy weights.")
print("Fields that matter more get higher weights, which tells the frontend")
print("to prompt users to fill them in.")

---
# Part 5: Reviewer Intelligence -- How the Customer Base Gets Modelled

Before generating personas, we try to understand the business's **actual customer base** by analysing their Google Reviews. This is the Reviewer Intelligence System -- the most research-heavy part of the pipeline.

The core insight: **Reviews are not representative.** Only ~1% of customers write reviews, and they skew extreme (very happy or very angry). The silent majority -- the 55-70% who never review -- are the most important customers to model.

The pipeline has 5 steps:

```
Google Reviews (raw)
    |
    v
1. Signal Extraction     extract_review_signals()
   (keyword counting,     -> AggregateReviewSignals
    no individual data)
    |
    v
2. Bias Correction        apply_bias_corrections()
   (extremity, platform,   -> BiasAdjustedSignals
    silent majority)
    |
    v
3. Silent Majority        estimate_silent_majority()
   (model who ISN'T        -> SilentMajorityProfile
    in the reviews)
    |
    v
4. Segment Builder        build_segments()
   (6 customer segments    -> CustomerSegmentProfile
    with proportions)
    |
    v
5. Persona Calibrator     calibrate_personas()
   (PersonaSpecs with      -> PersonaGenerationManifest
    fixed demographics)
```

In [ ]:
# We can't call the Google Places API here, so let's simulate what the
# pipeline looks like with realistic fake data.

from backend.reviewer_intelligence.review_signal_extractor import AggregateReviewSignals
from backend.reviewer_intelligence.bias_corrector import apply_bias_corrections
from backend.reviewer_intelligence.silent_majority_estimator import estimate_silent_majority
from backend.reviewer_intelligence.customer_segment_builder import build_segments
from backend.reviewer_intelligence.persona_calibrator import calibrate_personas
from backend.models.business import BusinessSnapshot

# Step 1: Simulated review signals (as if extracted from Google)
signals = AggregateReviewSignals(
    place_id="ChIJ_fake_id",
    total_review_count=147,
    average_rating=4.2,
    rating_distribution={5: 0.45, 4: 0.25, 3: 0.12, 2: 0.10, 1: 0.08},
    price_mention_frequency=0.35,     # 35% of reviews mention price
    value_positive_ratio=0.60,        # of those, 60% are positive about value
    loyalty_mention_frequency=0.22,
    switching_mention_frequency=0.08,
    tourist_mention_frequency=0.15,
    local_mention_frequency=0.40,
    tourist_ratio_estimate=0.18,
    detected_price_change=False,
    signal_confidence="medium",
)

print("=== Step 1: Raw Review Signals ===")
print(f"  Total reviews: {signals.total_review_count}")
print(f"  Average rating: {signals.average_rating}")
print(f"  Rating distribution: {signals.rating_distribution}")
print(f"  Price mentions: {signals.price_mention_frequency:.0%}")
print(f"  Tourist ratio: {signals.tourist_ratio_estimate:.0%}")
print(f"  Confidence: {signals.signal_confidence}")

In [ ]:
# Step 2: Bias correction
# Reviews are bimodally extreme and Google underrepresents negatives.
# This step compresses the tails and uplifts negative sentiment.

adjusted = apply_bias_corrections(signals)

print("=== Step 2: Bias-Adjusted Signals ===")
print()
print("  Corrections applied:")
for c in adjusted.corrections_applied:
    print(f"    - {c}")
print()
print(f"  Adjusted sentiment:")
print(f"    Positive: {adjusted.adjusted_positive_ratio:.2f} (was ~{signals.rating_distribution[5] + signals.rating_distribution[4]:.2f})")
print(f"    Negative: {adjusted.adjusted_negative_ratio:.2f} (was ~{signals.rating_distribution[1] + signals.rating_distribution[2]:.2f})")
print(f"    Moderate: {adjusted.adjusted_moderate_ratio:.2f}")
print()
print(f"  Estimated yearly customers: {adjusted.estimated_yearly_customers}")
print(f"  (That's {adjusted.estimated_yearly_customers}x from {signals.total_review_count} reviewers)")
print()
print(f"  Silent majority characterization:")
print(f"    Sentiment:         {adjusted.silent_majority_sentiment:.2f}")
print(f"    Price sensitivity: {adjusted.silent_majority_price_sensitivity:.2f}")
print(f"    Loyalty:           {adjusted.silent_majority_loyalty:.2f}")

In [ ]:
# Step 3: Silent majority estimation
# Models the 55-70% of customers who NEVER write reviews.

silent = estimate_silent_majority(adjusted, tourist_ratio=signals.tourist_ratio_estimate)

print("=== Step 3: Silent Majority Profile ===")
print(f"  Recommended swarm proportion: {silent.recommended_swarm_proportion:.0%}")
print(f"  Estimated satisfaction:        {silent.estimated_satisfaction:.2f}")
print(f"  Price sensitivity:             {silent.estimated_price_sensitivity:.2f}")
print(f"  Loyalty:                       {silent.estimated_loyalty:.2f}")
print(f"  Switching risk:                {silent.estimated_switching_risk:.2f}")
print()
print("  Persona guidance (what the LLM gets told about silent majority):")
print(f"  {silent.persona_guidance[:500]}...")
print()
print(f"  Confidence: {silent.confidence_note}")

In [ ]:
# Step 4: Customer segment builder
# Creates 6 segments with calibrated proportions.

business = BusinessSnapshot(
    name="La Tasca de Miguel",
    type="restaurant",
    description="Traditional Spanish tapas in the Gothic Quarter",
    customer_description="Local office workers, regulars, some tourists",
    location="Barcelona, Spain",
)

segment_profile = build_segments(adjusted, silent, business)

print("=== Step 4: Customer Segments ===")
print(f"  Confidence: {segment_profile.confidence_level}")
print(f"  Silent majority proportion: {segment_profile.silent_majority_proportion:.0%}")
print(f"  Most price-sensitive: {segment_profile.most_price_sensitive}")
print()
print(f"  {'Segment':<25} {'Proportion':>10} {'Price Sens':>12} {'Loyalty':>10}")
print("  " + "-" * 60)
for seg in segment_profile.segments:
    print(f"  {seg.name:<25} {seg.proportion:>9.0%} {seg.price_sensitivity:>11.2f} {seg.loyalty:>9.2f}")
print()
print(f"  Summary: {segment_profile.summary}")

In [ ]:
# Step 5: Persona calibration
# Converts segments into a PersonaGenerationManifest -- the specs the LLM must follow.

manifest = calibrate_personas(segment_profile, persona_count=15)

print(f"=== Step 5: Persona Generation Manifest ===")
print(f"  Total personas: {manifest.total_count}")
print(f"  Confidence: {manifest.confidence}")
print(f"  Based on: {manifest.based_on}")
print()
print(f"  Distribution: {manifest.distribution_summary}")
print()
print(f"  Anti-bias instructions (given to the LLM):")
print(f"  {manifest.anti_bias_instructions[:600]}...")
print()
print(f"  Key caveats: {manifest.key_caveats}")
print()
print(f"  === Individual Persona Specs ===")
print(f"  {'#':>3} {'Segment':<25} {'Age':>4} {'Income':<10} {'Visit Freq':<15} {'Price Sens':<12} {'Silent':>6}")
print("  " + "-" * 80)
for i, spec in enumerate(manifest.persona_specs):
    print(f"  {i+1:>3} {spec.segment:<25} {spec.age:>4} {spec.income_tier:<10} "
          f"{spec.visit_frequency:<15} {spec.price_sensitivity:<12} "
          f"{'yes' if spec.is_silent_majority else '-':>6}")

---
# Part 6: How Personas Are Generated

There are **two paths** for persona generation, and which one fires depends on whether the Reviewer Intelligence System produced a manifest.

### Path A: Manifest-Constrained (preferred)
When Google Reviews are available, we get a `PersonaGenerationManifest` with fixed specs for each persona. The LLM receives structured constraints (age, income, visit frequency, price sensitivity are LOCKED) and can only add narrative detail (name, backstory, personality, quirks).

**After the LLM responds, we overlay the fixed fields again** -- so even if the model ignores our instructions, the demographics stay accurate.

### Path B: Freeform (fallback)
When no reviews are available, we use the `persona_base.txt` prompt which asks Claude to generate diverse personas from scratch. This path is less grounded but still has anti-bias instructions.

Let's look at both prompts.

In [ ]:
# The freeform persona generation prompt (Path B -- fallback)

prompt_dir = PROJECT_ROOT / "backend" / "swarm" / "prompts"
persona_base = (prompt_dir / "persona_base.txt").read_text()

print("=== persona_base.txt (freeform generation) ===")
print(f"({len(persona_base)} characters)")
print()
print(persona_base)

In [ ]:
# The manifest-constrained prompt (Path A -- preferred)

persona_from_spec = (prompt_dir / "persona_from_spec.txt").read_text()

print("=== persona_from_spec.txt (manifest-constrained generation) ===")
print(f"({len(persona_from_spec)} characters)")
print()
print(persona_from_spec)

In [ ]:
# What the actual prompt looks like when assembled for our Spanish restaurant.
# This is the exact function from persona_generator.py.

from backend.swarm.persona_generator import _build_generation_prompt

assembled_prompt = _build_generation_prompt(
    business=business,
    persona_count=15,
    context_narrative=combined_narrative[:2000],  # truncated for display
)

print("=== Assembled Freeform Prompt (what Claude actually receives) ===")
print(f"({len(assembled_prompt)} characters)")
print()
# Show first 2000 chars
print(assembled_prompt[:2000])
if len(assembled_prompt) > 2000:
    print(f"\n... [{len(assembled_prompt) - 2000} more characters]")

In [ ]:
# What a generated persona looks like as a data structure.
# This is the PersonaProfile model from backend/models/persona.py.

from backend.models.persona import PersonaProfile

# Example persona (what Claude would return)
example_persona = PersonaProfile(
    name="Carlos",
    age=42,
    occupation="Insurance broker",
    visit_frequency="3x per week for lunch",
    avg_spend=14.50,
    personality="Cautious with money, loyal to routines, dislikes change. "
                "Will complain quietly to friends before confronting staff.",
    relationship_to_business="Has been coming every weekday for 2 years. "
                             "Sits at the same table by the window. Staff know his order.",
    quirk="Always checks the daily special price before ordering. "
          "Will switch to the menu del dia if the special is over 16 euros.",
    openness=0.3,
    conscientiousness=0.8,
    extraversion=0.4,
    agreeableness=0.6,
    neuroticism=0.5,
)

print("=== Example PersonaProfile ===")
print()
for field, value in example_persona.model_dump().items():
    if value is not None:
        print(f"  {field}: {value}")

---
# Part 7: The Simulation -- How Each Persona Is Interviewed

This is the core of Murmur. Each persona is interviewed **independently** (no cross-contamination to avoid herd mentality). The key design choices:

1. **Parallel execution** with `asyncio.gather()` -- all 15 personas fire at once
2. **Semaphore** limits concurrent API calls to 5 (respects Anthropic rate limits)
3. **60% minimum threshold** -- if fewer than 60% succeed, the whole simulation fails
4. **Each persona gets a unique system message** built from their profile

The interview prompt is the most carefully engineered piece. Version v0.2 added a critical "behavioral realism" section based on backtest failures where personas were too rational and principled.

In [ ]:
# The interview prompt template -- this is what each persona sees.

interview_prompt = (prompt_dir / "persona_interview.txt").read_text()

print("=== persona_interview.txt (v0.2) ===")
print(f"({len(interview_prompt)} characters)")
print()
print(interview_prompt)

In [ ]:
# What the assembled interview prompt looks like for a specific persona.
# This is the exact function from simulator.py.

from backend.swarm.simulator import _build_interview_prompt

filled_prompt = _build_interview_prompt(
    persona=example_persona,
    business=business,
    question="What if we raised all menu prices by 15%?",
    context_narrative=combined_narrative[:1500],  # truncated for display
)

print("=== Assembled Interview Prompt for Carlos (age 42, insurance broker) ===")
print(f"({len(filled_prompt)} characters)")
print()
print(filled_prompt[:3000])
if len(filled_prompt) > 3000:
    print(f"\n... [{len(filled_prompt) - 3000} more characters]")

In [ ]:
# What a persona response looks like.
# This is what Claude returns for each persona interview.
# (Example -- not a live API call)

example_response = {
    "persona_name": "Carlos",
    "reaction": "Fifteen percent? That is a lot. My menu del dia is already 13.50, "
                "so that would push it past 15 euros. I would notice immediately. "
                "I would probably still come because I have been eating here for years "
                "and I like the routine, but I would switch to the cheaper options more "
                "often. And honestly, I would start checking prices at the place two "
                "streets down that I have been ignoring.",
    "reasoning": "I am a creature of habit and this place is part of my daily routine, "
                 "so I would not leave right away. But I watch every euro and 15% is "
                 "not something I would just absorb. My loyalty buys them time, not "
                 "a free pass.",
    "sentiment": -0.35,
    "raw": {
        "reaction": "...",
        "reasoning": "...",
        "sentiment": -0.35,
    }
}

print("=== Example Persona Response ===")
print()
print(f"  Persona: {example_response['persona_name']}")
print(f"  Sentiment: {example_response['sentiment']}")
print()
print(f"  Reaction:")
print(f"    {example_response['reaction']}")
print()
print(f"  Reasoning:")
print(f"    {example_response['reasoning']}")

In [ ]:
# The parallel execution architecture from simulator.py.
# This is the actual code (simplified for clarity).

print("=== Simulation Execution Architecture ===")
print()
print("""
async def run_simulation(personas, business, question, ...):
    # Rate-limit concurrent API calls
    semaphore = asyncio.Semaphore(5)  # max 5 at once

    async def interview_with_progress(persona):
        result = await _interview_persona(
            client, semaphore, persona, business, question, ...
        )
        return result

    # Fire ALL personas in parallel (semaphore handles concurrency)
    results = await asyncio.gather(
        *[interview_with_progress(p) for p in personas],
        return_exceptions=True,   # one failure doesn't kill the rest
    )

    # Separate successes from failures
    successes = [r for r in results if not isinstance(r, Exception)]
    failures = [r for r in results if isinstance(r, Exception)]

    # 60% must succeed or the whole simulation fails
    min_required = int(len(personas) * 0.6)
    if len(successes) < min_required:
        raise RuntimeError("Too many persona failures")

    return successes
""")

print("For 15 personas with semaphore=5:")
print("  - 3 batches of 5 fire concurrently")
print("  - Each call takes ~3-5 seconds")
print("  - Total interview phase: ~9-15 seconds")
print("  - Minimum 9/15 must succeed")

---
# Part 8: Aggregation -- Turning 15 Voices into One Report

After all persona interviews complete, the aggregator reads every response and produces a structured report. This is another Claude API call with a carefully designed prompt.

The aggregator's job is to sound like **a trusted advisor who just finished talking to real customers** -- never like an AI report.

In [ ]:
# The aggregation prompt template

aggregation_prompt = (prompt_dir / "aggregation.txt").read_text()

print("=== aggregation.txt (v0.2) ===")
print(f"({len(aggregation_prompt)} characters)")
print()
print(aggregation_prompt)

---
# Part 9: Impact Estimation + Caveats

After aggregation, two more systems run:

### Impact Estimator
Converts sentiment scores into **quantitative revenue/retention estimates with confidence intervals**. Based on Prof. Uri Simonsohn's decision framework:
- If worst case is still good -> **proceed**
- If best case is still bad -> **avoid**
- If best case is good but worst case is bad -> **test first**

### Caveat Generator
Pattern-matches the question text against known pitfalls from A/B testing research and generates warnings. Some caveats fire on every simulation (like "this is not causation"). Others only fire when triggered (like regression-to-the-mean warnings when the user says "sales are down").

In [ ]:
# Impact estimation with realistic simulated responses

from backend.impact.estimator import estimate_impact

# Simulate 15 persona responses with varied sentiments
simulated_responses = [
    {"persona_name": "Carlos", "sentiment": -0.35},     # silent regular, price-sensitive
    {"persona_name": "Maria", "sentiment": -0.50},      # silent occasional, would leave
    {"persona_name": "Jordi", "sentiment": 0.20},       # loyal fan, accepts it
    {"persona_name": "Elena", "sentiment": -0.60},      # value seeker, angry
    {"persona_name": "Pere", "sentiment": -0.15},       # silent regular, reluctant accept
    {"persona_name": "Ana", "sentiment": 0.40},         # loyal fan, supportive
    {"persona_name": "Miguel", "sentiment": -0.25},     # silent occasional, annoyed
    {"persona_name": "Rosa", "sentiment": -0.10},       # silent regular, neutral
    {"persona_name": "David", "sentiment": 0.00},       # tourist, doesn't care
    {"persona_name": "Lucia", "sentiment": -0.45},      # silent occasional, would reduce
    {"persona_name": "Pablo", "sentiment": -0.30},      # value seeker, unhappy
    {"persona_name": "Carmen", "sentiment": 0.10},      # silent regular, accepts grudgingly
    {"persona_name": "Tomas", "sentiment": -0.70},      # frustrated customer, furious
    {"persona_name": "Isabel", "sentiment": -0.20},     # silent regular, grumbles
    {"persona_name": "Andres", "sentiment": 0.05},      # tourist, indifferent
]

question = "What if we raised all menu prices by 15%?"
impact = estimate_impact(simulated_responses, question)

print("=== Impact Estimation for 15% Price Increase ===")
print()
print(f"  Revenue impact: {impact.revenue.point_estimate_pct:+.1f}%")
print(f"  Confidence interval: [{impact.revenue.ci_low_pct:+.1f}%, {impact.revenue.ci_high_pct:+.1f}%]")
print(f"  Confidence level: {impact.revenue.confidence_level}")
print()
print(f"  Customer retention:")
print(f"    Would stay:       {impact.customers_likely_stay:>2}/15")
print(f"    Would reduce:     {impact.customers_likely_reduce:>2}/15")
print(f"    Would leave:      {impact.customers_likely_leave:>2}/15")
print(f"    Retention rate:   {impact.retention_rate_pct:.0f}%")
print()
print(f"  DECISION: {impact.decision.upper()}")
print(f"  {impact.decision_reasoning}")
print()
print(f"  Framework: {impact.decision_framework}")
print()
print(f"  Worst case: {impact.worst_case_summary}")
print(f"  Best case:  {impact.best_case_summary}")
print(f"  Most likely: {impact.most_likely_summary}")

In [ ]:
# Caveats -- what warnings fire for this simulation?

from backend.swarm.caveats import generate_caveats

caveats = generate_caveats(
    business=business,
    question=question,
    persona_count=15,
    success_count=15,
    variant_a=None,
    variant_b=None,
    review_signals=signals,  # from the reviewer intelligence step
)

print(f"=== Caveats Generated ({len(caveats)} total) ===")
print()
for i, c in enumerate(caveats):
    print(f"  {i+1}. [{c.severity.upper()}] {c.title}")
    print(f"     Type: {c.type}")
    print(f"     Source: {c.source}")
    print(f"     {c.message[:200]}")
    print()

---
# Part 10: The Final Result -- What Gets Stored and Shown to the User

Everything comes together into a `SimulationResult` stored in Supabase. Here is what the complete data structure looks like:

In [ ]:
# What the final result looks like (example -- not from a live API call)

from backend.models.simulation import SimulationResult, Theme, StandoutVoice
from uuid import uuid4
from datetime import datetime, timezone

example_result = SimulationResult(
    id=uuid4(),
    simulation_id=uuid4(),
    summary="Most of your regulars would absorb a 15% price increase grudgingly, "
            "but your value seekers and occasional visitors would start looking elsewhere. "
            "Your loyalists would stay but your Monday lunch crowd is at risk.",
    recommendation="Consider a smaller increase (8-10%) or a tiered approach -- "
                   "raise dinner prices by 15% but keep the lunch menu del dia under 15 euros. "
                   "Your weekday regulars are the most sensitive group and the hardest to replace.",
    confidence_score="medium",
    confidence_reasoning="Mixed signals: strong consensus among regulars (would stay but complain) "
                         "but high variance among occasional visitors. Business profile was detailed "
                         "which helps, but price sensitivity is hard to simulate accurately.",
    themes=[
        Theme(label="The grudging regulars", summary="Would stay out of habit but would trade down to cheaper menu items and watch prices more carefully", count=6),
        Theme(label="The at-risk occasionals", summary="Would reduce visits or try competitors they have been ignoring", count=4),
        Theme(label="The loyal core", summary="Would accept the increase and continue their routine, viewing it as still worth it", count=3),
        Theme(label="The indifferent", summary="Tourists and one-time visitors who would not even notice", count=2),
    ],
    standout_voices=[
        StandoutVoice(persona_name="Carlos", quote="I would probably still come but I would start checking the place two streets down"),
        StandoutVoice(persona_name="Elena", quote="Fifteen percent is a lot when you come three times a week. That is 50 euros a month more."),
        StandoutVoice(persona_name="Jordi", quote="Honestly, if the food stays this good, I do not care. I come here for Miguel's cooking, not the prices."),
    ],
    raw_output={},
    created_at=datetime.now(timezone.utc),
)

print("=== Final Simulation Result ===")
print()
print(f"HEADLINE: {example_result.summary}")
print()
print(f"CONFIDENCE: {example_result.confidence_score}")
print(f"  {example_result.confidence_reasoning}")
print()
print("THEMES:")
for t in example_result.themes:
    print(f"  [{t.count} personas] {t.label}: {t.summary}")
print()
print("STANDOUT VOICES:")
for v in example_result.standout_voices:
    print(f'  {v.persona_name}: "{v.quote}"')
print()
print(f"RECOMMENDATION: {example_result.recommendation}")

---
# Part 11: What Gets Stored in Supabase

Every simulation run persists everything for audit, debugging, and future backtesting:

| Table | What it stores | Row count per simulation |
|-------|---------------|------------------------|
| `simulations` | Question, status, business snapshot, prompt version, context data, caveats, impact data, reviewer intelligence | 1 |
| `personas` | All generated persona profiles (full JSON) | 15 |
| `persona_responses` | Each persona's reaction, reasoning, sentiment, raw output | 15 |
| `simulation_results` | Aggregated summary, themes, standout voices, confidence, recommendation | 1 |
| `real_outcomes` | What actually happened (submitted later by the user) | 0-1 |

**Total per simulation: 32 rows across 4 tables, plus the simulation row itself.**

The `real_outcomes` table is the critical feedback loop -- when users report what actually happened, we can retrain the ML model and improve prompt accuracy.

---
# Part 12: API Cost Per Simulation

Each simulation makes multiple Claude API calls. Here is the breakdown:

| Phase | API Calls | Model | Tokens (approx) |
|-------|-----------|-------|-----------------|
| Context orchestrator | 1 | claude-sonnet | ~2K in, ~500 out |
| Context tools (web search, review analyzer) | 0-2 | claude-sonnet | ~3K in, ~500 out each |
| Context relevance filter | 0-1 | claude-sonnet | ~4K in, ~1K out |
| Persona generation | 1 | claude-sonnet | ~3K in, ~4K out |
| Persona interviews | 15 | claude-sonnet | ~2K in, ~500 out each |
| Aggregation | 1 | claude-sonnet | ~8K in, ~2K out |
| **Total** | **~20 calls** | | **~55K tokens** |

At Sonnet pricing ($3/MTok input, $15/MTok output), each simulation costs roughly **$0.15-0.30**.

---

# Summary

The simulation pipeline is a 6-phase system that takes a business survey, enriches it with research + live market data, builds a calibrated customer model from reviews, generates structured personas, interviews them independently in parallel, and synthesizes the results into a human-readable report with quantitative impact estimates and research-backed caveats.

**What makes it different from generic AI personas:**
1. Review bias correction (extremity compression, platform adjustment, silent majority modeling)
2. Structured demographic anchoring from peer-reviewed research (NeurIPS 2025 / ICLR 2024)
3. Behavioral realism instructions (stated vs revealed preference gap, friction effects)
4. Confidence intervals with a decision framework (not just "the AI says yes")
5. Every run is logged for backtesting and model improvement